[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astheeggeggs/testing_colab/blob/main/LDSC_Practical_1_Colab.ipynb)

# LD-score regression (LDSC) practical 1: SNP Heritability $\left(h_{\text{SNP}}^2\right)$

Before we get cracking on the practical, we need to get our environment set up, and all the required R packages installed.

## 1) Environment setup / required packages

This Colab notebook uses an **R runtime**. We install the packages needed for the practical from CRAN and GitHub.

Packages used here:
- `data.table`
- `dplyr`
- `remotes`
- `GenomicSEM`


In [ ]:
# System dependencies that are often useful for R package installation
# (safe to run even if some packages are already present)
system("apt-get update -qq")
system("apt-get install -y -qq libcurl4-openssl-dev libxml2-dev libssl-dev")
system('python -m pip -q install gdown')
system('gdown --folder "https://drive.google.com/drive/folders/1BIBaoLlodLG_ni9DASh_bw-3RzUZSSJe?usp=sharing" -O /content/LDSC_Practical_1')
root_dir <- "/content/LDSC_Practical_1"
sumstat_dir <- file.path(root_dir, "EUR")
ref_data_dir <- file.path(root_dir, "EUR", "eur_w_ld_chr")

In [ ]:
# Install CRAN packages
install.packages(
  c("data.table", "dplyr", "remotes"),
  repos = "https://cloud.r-project.org"
)

# Install GenomicSEM from GitHub
# GenomicSEM is maintained on GitHub and its README recommends installation from GitHub.
remotes::install_github("GenomicSEM/GenomicSEM", upgrade = "never")

library(data.table)
library(dplyr)
library(remotes)
library(GenomicSEM)


Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Skipping install of 'GenomicSEM' from a github remote, the SHA1 (123ec965) has not changed since last install.
  Use `force = TRUE` to force installation


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%



Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [ ]:
dir("/content/LDSC_Practical_1")
setwd("/content/LDSC_Practical_1")

[1] "EUR"                                "LDSC_Practical1_h2_SNP.R"          
[3] "LDSC_Practical1_h2snp_MS2025.pdf"   "Neale_Singh_LDSC_2025_Boulder.pdf" 
[5] "Neale_Singh_LDSC_2025_Boulder.pptx"

### ✅ Load packages

If the install cell ran successfully, load the packages here.


In [ ]:
library(data.table)
library(dplyr)
library(GenomicSEM)

sessionInfo()


Only TWO Primary Steps to Run LDSC
We are using {GenomicSEM} library in R to run LDSC
1. Munge the summary statistics: munge()
munge = convert raw data from one form to another
2. Run LD-Score Regression: ldsc()

The summary statistics files input to munge() at a minimum
need to contain five pieces of information:
1.The rsID of the SNP.
2.An A1 allele column, indicating the effect allele.
3.An A2 allele column, indicating the non-effect allele.
4.A signed (+/-) effect column.
5.The p-value associated with this effect.

The munge() function takes 6 arguments:
1.files: The name of the summary statistics files
2.hm3: The name of the reference file. Here we use Hapmap 3 SNPs.
3.trait.names: The trait names that will be used to name the saved files
4.N: The sample sizes associated with the traits.
5.info.filter: INFO filter. Package default is to retain SNPs with INFO > 0.9.
6.maf.filter: MAF filter. Package default is to retain SNPs with MAF > 0.01

The ldsc() function takes 6 arguments:
1.traits: a vector of file names/paths to files which point to the munged sumstats.
2.sample.prev: A vector of sample prevalences of length equal to the number of
traits. If the trait is continuous, the values should equal NA.
3.population.prev: A vector of population prevalences. If the trait is continuous
the values should equal NA.
4. ld: A folder of LD scores used as the independent variable in LDSC
5. wld: A folder of LDSC weights (Typically the same folder as specified for the ld
argument)
6. trait.names: The trait names.

## 2) Set up the working directory in Colab

Unlike the original local practical, Colab does not have your course folder on disk.

The code below assumes you have a folder with this structure:

```text
LDSC_Practical_1/
  EUR/
    w_hm3.snplist
    eur_w_ld_chr/
    CHR22only_Meta-analysis_Locke_et_al+UKBiobank_2018_UPDATED.txt.gz
    Meta-analysis_Locke_et_al+UKBiobank_2018_UPDATED.txt.gz
    Munged_BMI_Meta-analysis_Locke_et_al+UKBiobank_2018_for_LDSC.sumstats.gz
    BMI.sumstats.gz
```


In [ ]:
# Set directories
sumstat_dir  <- "EUR/"
ref_data_dir <- "EUR/eur_w_ld_chr/"

# Quick check
getwd()
dir(ref_data_dir)

[1] "/content/LDSC_Practical_1"

[1] "1.l2.ldscore.gz"     "1.l2.M_5_50"         "10.l2.ldscore.gz"   
 [4] "10.l2.M_5_50"        "11.l2.ldscore.gz"    "11.l2.M_5_50"       
 [7] "12.l2.ldscore.gz"    "12.l2.M_5_50"        "13.l2.ldscore.gz"   
[10] "13.l2.M_5_50"        "14.l2.ldscore.gz"    "14.l2.M_5_50"       
[13] "15.l2.ldscore.gz"    "15.l2.M_5_50"        "16.l2.ldscore.gz"   
[16] "16.l2.M_5_50"        "17.l2.ldscore.gz"    "17.l2.M_5_50"       
[19] "18.l2.ldscore.gz"    "18.l2.M_5_50"        "19.l2.ldscore.gz"   
[22] "19.l2.M_5_50"        "2.l2.ldscore.gz"     "2.l2.M_5_50"        
[25] "20.l2.ldscore.gz"    "20.l2.M_5_50"        "21.l2.ldscore.gz"   
[28] "21.l2.M_5_50"        "22.l2.ldscore.gz"    "22.l2.M_5_50"       
[31] "3.l2.ldscore.gz"     "3.l2.M_5_50"         "4.l2.ldscore.gz"    
[34] "4.l2.M_5_50"         "5.l2.ldscore.gz"     "5.l2.M_5_50"        
[37] "6_old.l2.ldscore.gz" "6_old.l2.M_5_50"     "6.l2.ldscore.gz"    
[40] "6.l2.M_5_50"         "7.l2.ldscore.gz"     "7.l2.M_5_50"        
[43] "8.l2.ldscore.gz"     "8.l2.M_5_50"         "9.l2.ldscore.gz"    
[46] "9.l2.M_5_50"         "README"              "w_hm3.snplist"

The reference directory includes the HapMap3 (HM3) SNP IDs and their pre-computed LD scores.

For LDSC to work as expected, the included SNPs must be QCed. To ensure that, LDSC is often done using only the HM3 SNPs because
- HM3 SNPs are expected to be common variants (MAF > 0.01).
- HM3 SNPs typically have high imputation quality.

So, if the GWAS sum stats do not have information on these QC metrics, we may run LDSC using only the HM3 SNPs.

Also, pre-computed LD scores for these SNPs (in European ancestry) were provided with the original LDSC program - so that makes it much easier to run LDSC in European ancestry!

Examine the HM3 SNP file using the code that you have already run.

## 3) HapMap3 SNPs and LD scores in European ancestry

We first inspect the HapMap3 SNP list used by LDSC.


### HapMap3 SNPs

**Question:** How many SNPs are there in the HapMap3 reference file?


In [ ]:
hm3_file <- file.path(ref_data_dir, "w_hm3.snplist")

# Take a look at the first few lines
cat(system(paste("head", hm3_file), intern = TRUE), sep = "\n")

# Count the number of lines
cat(system(paste("wc -l", hm3_file), intern = TRUE), sep = "\n")

SNP	A1	A2
rs3094315	G	A
rs3131972	A	G
rs3131969	A	G
rs1048488	C	T
rs3115850	T	C
rs2286139	C	T
rs12562034	A	G
rs4040617	G	A
rs2980300	T	C
1217312 EUR/eur_w_ld_chr//w_hm3.snplist


Answer: There are 1,217,311 SNPs included in the HM3 reference file.

Note that there are 1,217,312 lines in w_hm3.snplist file, but this number of lines also includes the first line with column names. So, we subtract 1 to get the number of rows (= SNPs) in the dataset.



We will still restrict the LDSC analyses to the HapMap3 SNPs in this practical because we have the pre-computed LD scores for these SNPs only. Alternatively, one may compute LD scores for all QC-ed SNPs in the GWAS sample or an external LD reference panel.


We may specify additional QC filters (imputation R2 [info_cutoff <- 0.9] and minor allele frequency [maf_cutoff  <- 0.01]).

We will drop the SNPs with imputation R2 (INFO) < 0.90 or MAF < 0.01.
These filters will be applied only if the GWAS sum stats include information on these QC metrics. Otherwise, these filters are ignored.

## 4) QC filters for LDSC


In [ ]:
info_cutoff <- 0.9
maf_cutoff  <- 0.01


We will perform the analyses in two steps:

1. Munge the data - munge()
This step "cleans" your GWAS sum stats file to select (and rename if needed) the required columns.
This step also filters the SNPs to keep only the SNP IDs listed in the HapMap3 reference file. This filtering is important because all SNPs retained in the munged data must have their LD scores computed and stored in the reference directory.
If the sum stats dataset includes QC metrics, this step will also apply the QC filters.

At the end of this step, the munged dataset will be saved in the specified location.

2. LD score regression - ldsc()
This is the main LDSC analysis using the munged dataset.
The results are saved as a log file in the specified location.

## 5) BMI example: subset to chromosome 22

For the tutorial, we practice `munge()` using only chromosome 22.

LDSC needs these column names in the munged data:
SNP -- SNP identifier (e.g., rsID)
N -- sample size (which may vary from SNP to SNP).
BETA -- Beta. (or Z-score, or OR) **Sign with respect to A1** (Warning: Possible gotcha!)
A1 -- first allele (effect allele)
A2-- second allele (other allele)
P-Value
Any QC metrics, e.g., INFO or MAF

Examine the BMI gwas sum stats (the chr 22 subset) prior to munging.

Read the code in script lines 56 to 70, and Run these lines of code.

In this code, we want to verify which allele is the "effect allele" with respect to which the association statistic BETA is signed (positive or negative).

Look at the columns in the BMI sum stats.

Are the column names informative about the "effect allele"?

In this case, YES, as the effect allele is labeled "Tested_Allele"!

This is helpful, as we do not have a README file.

If the column names are uninformative and there is no README file, do not hesitate to verify with the corresponding author!

Using the following block of code, we rename the columns to what LDSC expects and save the file with the new column names.
Note that renaming the columns is not required if the sum stats file has standard column names that the program can interpret correctly - which is the case here. However, that may not always be the case.

Some GWAS sum stats may report the "effect allele frequency" rather than MAF. In that case, we'd need to convert the effect allele frequency to MAF.

Run the script lines 71 to 100.

Now, munge the BMI sum stats using the following code.

Read the code in script lines 101 to 120, and Run these lines of code.

Examine the log file and answer the following questions

Which column is interpreted as the effect allele (A1)?

Answer: The A1 column, which is the correct column.

Remember that we renamed the "Tested_Allele" column as "A1"

How many SNPs were present in the raw sum stats file?

Answer: There were 29,991 SNPs present in the raw sum stats file.

How many SNPs were dropped because these are not present in the reference HapMap3 file?

How many SNPs were dropped due to low imputation quality (i.e., INFO)?

Answer: None, as there was no INFO column in the raw dataset.

How many SNPs were dropped due to low or missing MAF?

Answer: 1 SNP was dropped due to low or missing MAF.


Usually, there will be few HapMap3 SNPs with MAF < 0.01 - which is why only one SNP was dropped here.

How many SNPs are left in the munged sum stats file?

Answer: 14,382 SNPs are left in the munged sum stats file.

Now we are done munging the BMI sum stats (chr 22 subset).

We have already munged the full sum stats file (for all chromosomes).

Using the code below, examine and compare the raw and the munged sum stats.

Run the script lines 121 to 140.


Look at the head (i.e., the first few lines) of the munged sum stats.

What are the columns included in the munged sum stats?

Answer

SNP, N, Z, A1, A2
Now run the LDSC analysis of BMI using the munged genome-wide sum stats and the following lines of code.

Run the script lines 141 to 160.

Note that we are running LDSC using `GenomicSEM` library, which is primarily meant for multivariate analyses.
So, we expect the following warning: "Our version of ldsc requires 2 or more traits. Please include an additional trait."
This is okay. We will run multivariate LDSC in the next session.

The output log file also includes genetic correlations. These are not relevant to the current univariate analyses but will be covered in the next session.

Examine the log (printed to your R console)

What are the Mean Chi^2 and the Lambda GC? Do these suggest an inflation of GWAS statistics?

Answer

Mean Chi^2 = 3.9345.
Lambda GC = 2.7889.

Both are >1 and so suggest inflation of GWAS statistics.

However, this inflation may be due to true genetic associations (polygenicity and large enough sample size) and potential confounding (e.g., population stratification).

What is the LDSC intercept? Does it suggest significant bias(es) in the BMI GWAS results?

Answer

LDSC intercept = 1.0202 (S.E. 0.0277).

As the LDSC intercept is not significantly greater than 1, it suggests that there was no significant confounding (population stratification) in the GWAS.

What is the estimated SNP heritability (h2_snp) of BMI?

Answer

h2_snp of BMI = 0.2091 (S.E. 0.0063)


Congratulations! You successfully ran the LDSC analyses of BMI.


Please note that LDSC uses the provided [munged] sum stats to estimate the heritability/variance per SNP. The program then uses the estimate of the per-SNP heritability to compute the total variance across all SNPs in the genome [i.e., the SNP heritability].

If the sum stats used to run LDSC analyses are NOT a random subset of all SNPs across the genome, the estimated per-SNP variance and, thus, the SNP-based heritability will be biased.


In [ ]:
bmi_file <- file.path(
  sumstat_dir,
  "CHR22only_Meta-analysis_Locke_et_al+UKBiobank_2018_UPDATED.txt.gz"
)

cat(system(paste0("zcat ", bmi_file, " | head"), intern=TRUE), sep="\n")


CHR	POS	SNP	Tested_Allele	Other_Allele	BETA	SE	P	N	MAF
22	28273339	rs1000112	C	G	-0.0051	0.0031	0.097	679794	0.08326
22	36890105	rs1000427	A	G	-0.001	0.0028	0.73	692567	0.1118
22	24026845	rs1000470	A	C	0.0017	0.0029	0.55	675111	0.1074
22	20202729	rs1000539	A	G	-0.0025	0.0019	0.19	669369	0.2933
22	26831077	rs1000815	A	G	-0.0077	0.0017	9.2e-06	692285	0.4701
22	22051709	rs10009	A	G	0.0066	0.0018	0.00018	691323	0.3985
22	26403488	rs1001022	T	C	-0.0088	0.0055	0.11	549516	0.03438
22	34131736	rs1001213	A	G	0.0042	0.0038	0.27	690008	0.05958
22	33763247	rs1001223	T	C	0.005	0.0045	0.26	665582	0.04613


### Rename columns as needed

Renaming is not essential if LDSC can already interpret the column names, but it is useful when the summary statistics use non-standard names.


In [ ]:
bmi_GWAS <- fread(bmi_file, header = TRUE, data.table = FALSE)
str(bmi_GWAS)

bmi_GWAS <- dplyr::rename(
  bmi_GWAS,
  A1 = Tested_Allele,
  A2 = Other_Allele
)

head(bmi_GWAS)


'data.frame':	29991 obs. of  10 variables:
 $ CHR          : int  22 22 22 22 22 22 22 22 22 22 ...
 $ POS          : int  28273339 36890105 24026845 20202729 26831077 22051709 26403488 34131736 33763247 36630949 ...
 $ SNP          : chr  "rs1000112" "rs1000427" "rs1000470" "rs1000539" ...
 $ Tested_Allele: chr  "C" "A" "A" "A" ...
 $ Other_Allele : chr  "G" "G" "C" "G" ...
 $ BETA         : num  -0.0051 -0.001 0.0017 -0.0025 -0.0077 0.0066 -0.0088 0.0042 0.005 -0.004 ...
 $ SE           : num  0.0031 0.0028 0.0029 0.0019 0.0017 0.0018 0.0055 0.0038 0.0045 0.0035 ...
 $ P            : num  9.7e-02 7.3e-01 5.5e-01 1.9e-01 9.2e-06 1.8e-04 1.1e-01 2.7e-01 2.6e-01 2.5e-01 ...
 $ N            : int  679794 692567 675111 669369 692285 691323 549516 690008 665582 677031 ...
 $ MAF          : num  0.0833 0.1118 0.1074 0.2933 0.4701 ...


,CHR,POS,SNP,A1,A2,BETA,SE,P,N,MAF
,<int>,<int>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<int>,<dbl>
1,22,28273339,rs1000112,C,G,-0.0051,0.0031,9.7e-02,679794,0.08326
2,22,36890105,rs1000427,A,G,-0.0010,0.0028,7.3e-01,692567,0.11180
3,22,24026845,rs1000470,A,C,0.0017,0.0029,5.5e-01,675111,0.10740
4,22,20202729,rs1000539,A,G,-0.0025,0.0019,1.9e-01,669369,0.29330
5,22,26831077,rs1000815,A,G,-0.0077,0.0017,9.2e-06,692285,0.47010
6,22,22051709,rs10009,A,G,0.0066,0.0018,1.8e-04,691323,0.39850


In [ ]:
bmi_updated_file <- file.path(sumstat_dir, "BMI_GWAS_for_LDSC.txt")

fwrite(
  bmi_GWAS,
  file = bmi_updated_file,
  sep = "\t",
  col.names = TRUE,
  row.names = TRUE,
  quote = FALSE
)

cat(system(paste0("head ", bmi_updated_file), intern=TRUE), sep='\n')


	CHR	POS	SNP	A1	A2	BETA	SE	P	N	MAF
1	22	28273339	rs1000112	C	G	-0.0051	0.0031	0.097	679794	0.08326
2	22	36890105	rs1000427	A	G	-0.001	0.0028	0.73	692567	0.1118
3	22	24026845	rs1000470	A	C	0.0017	0.0029	0.55	675111	0.1074
4	22	20202729	rs1000539	A	G	-0.0025	0.0019	0.19	669369	0.2933
5	22	26831077	rs1000815	A	G	-0.0077	0.0017	9.2e-06	692285	0.4701
6	22	22051709	rs10009	A	G	0.0066	0.0018	0.00018	691323	0.3985
7	22	26403488	rs1001022	T	C	-0.0088	0.0055	0.11	549516	0.03438
8	22	34131736	rs1001213	A	G	0.0042	0.0038	0.27	690008	0.05958
9	22	33763247	rs1001223	T	C	0.005	0.0045	0.26	665582	0.04613


### Munge

Provide the file path and name to save the munged summary statistics.


In [ ]:
library(GenomicSEM)
bmi_munged <- file.path(sumstat_dir, "munged_BMI_GWAS_chr22_for_LDSC")

munge(
  files = bmi_updated_file,
  hm3 = hm3_file,
  trait.names = bmi_munged,
  info.filter = info_cutoff,
  maf.filter = maf_cutoff
)


Warning message:
“replacing previous import ‘gdata::nobs’ by ‘lavaan::nobs’ when loading ‘GenomicSEM’”
Warning message:
“replacing previous import ‘gdata::last’ by ‘data.table::last’ when loading ‘GenomicSEM’”
Warning message:
“replacing previous import ‘gdata::first’ by ‘data.table::first’ when loading ‘GenomicSEM’”
Warning message:
“replacing previous import ‘gdata::resample’ by ‘R.utils::resample’ when loading ‘GenomicSEM’”
Warning message:
“replacing previous import ‘gdata::env’ by ‘R.utils::env’ when loading ‘GenomicSEM’”
Warning message:
“replacing previous import ‘data.table::first’ by ‘dplyr::first’ when loading ‘GenomicSEM’”
Warning message:
“replacing previous import ‘data.table::between’ by ‘dplyr::between’ when loading ‘GenomicSEM’”
Warning message:
“replacing previous import ‘plyr::summarize’ by ‘dplyr::summarize’ when loading ‘GenomicSEM’”
Warning message:
“replacing previous import ‘plyr::desc’ by ‘dplyr::desc’ when loading ‘GenomicSEM’”
Warning message:
“replacing previ

The munging of 1 summary statistics started at 2026-05-22 12:41:27.490894
Reading in reference file
Reading summary statistics for EUR//BMI_GWAS_for_LDSC.txt. Please note that this step usually takes a few minutes due to the size of summary statistic files.
All files loaded into R!
Munging file: EUR//BMI_GWAS_for_LDSC.txt
Interpreting the SNP column as the SNP column.
Interpreting the A1 column as the A1 column.
Interpreting the A2 column as the A2 column.
Interpreting the BETA column as the effect column.
Interpreting the P column as the P column.
Interpreting the N column as the N column.
Interpreting the MAF column as the MAF column.
Interpreting the SE column as the SE column.
Merging file:EUR//BMI_GWAS_for_LDSC.txt with the reference file:EUR/eur_w_ld_chr//w_hm3.snplist
29991 rows present in the full EUR//BMI_GWAS_for_LDSC.txt summary statistics file.
15608 rows were removed from the EUR//BMI_GWAS_for_LDSC.txt summary statistics file as the rs-ids for these rows were not present i

## 6) LDSC

The full BMI summary statistics are already munged for the practical. We compare the raw file with the munged file and then run LDSC.


In [ ]:
raw_bmi    <- file.path(sumstat_dir, "Meta-analysis_Locke_et_al+UKBiobank_2018_UPDATED.txt.gz")
munged_bmi <- file.path(sumstat_dir, "Munged_BMI_Meta-analysis_Locke_et_al+UKBiobank_2018_for_LDSC.sumstats.gz")

cat(system(paste0("zcat ", raw_bmi, " | head"), intern=TRUE), sep='\n')
cat(system(paste0("zcat ", munged_bmi, " | head"), intern=TRUE), sep='\n')

cat(system(paste0("zcat ", raw_bmi, " | wc -l"), intern=TRUE), sep='\n')
cat(system(paste0("zcat ", munged_bmi, " | wc -l"), intern=TRUE), sep='\n')


CHR	POS	SNP	Tested_Allele	Other_Allele	Freq_Tested_Allele_in_HRS	BETA	SE	P	N
7	92383888	rs10	A	C	0.06431	 0.0013	0.0042	 7.5e-01	598895
12	126890980	rs1000000	A	G	0.2219	 0.0001	0.0021	 9.6e-01	689928
4	21618674	rs10000010	T	C	0.5086	-0.0001	0.0016	 9.4e-01	785319
4	1357325	rs10000012	C	G	0.8634	 0.0047	0.0025	 5.7e-02	692463
4	37225069	rs10000013	A	C	0.7708	-0.0061	0.0021	 3.3e-03	687856
4	84778125	rs10000017	T	C	0.2284	 0.0041	0.0021	 4.8e-02	686123
3	183635768	rs1000002	T	C	0.4884	-0.0055	0.0017	 1.3e-03	692520
4	95733906	rs10000023	T	G	0.5817	-0.0047	0.0018	 7.2e-03	676691
4	156176217	rs10000027	C	G	0.771	-0.0013	0.0023	 5.7e-01	525093
SNP	N	Z	A1	A2
rs1000000	689928	0.0501535834647337	A	G
rs10000010	785319	0.0752698620998299	C	T
rs1000002	692520	-3.21597976078811	T	C
rs10000023	676691	2.68744944715147	G	T
rs1000003	690549	-1.20035885803086	G	A
rs10000033	677562	1.96859166918659	C	T
rs10000037	691768	0.318639363964375	A	G
rs10000041	689797	-0.240426031142308	G	T
rs1000007	688538	-1.

In [ ]:
bmi_ldsc_out <- file.path(sumstat_dir, "BMI")

ldsc(
  traits = munged_bmi,
  sample.prev = NA,
  population.prev = NA,
  ld = ref_data_dir,
  wld = ref_data_dir,
  trait.names = "BMI",
  ldsc.log = bmi_ldsc_out
)


Multivariate ld-score regression of 1 traits (EUR//Munged_BMI_Meta-analysis_Locke_et_al+UKBiobank_2018_for_LDSC.sumstats.gz) began at: 2026-05-22 12:43:11.13951


Warning message in ldsc(traits = munged_bmi, sample.prev = NA, population.prev = NA, :
“Our version of ldsc requires 2 or more traits. Please include an additional trait.”


Reading in LD scores
Read in summary statistics [1/1] from: EUR//Munged_BMI_Meta-analysis_Locke_et_al+UKBiobank_2018_for_LDSC.sumstats.gz
Out of 1019839 SNPs, 1014985 remain after merging with LD-score files
Removing 21 SNPs with Chi^2 > 795.64; 1014964 remain
Estimating heritability [1/1] for: EUR//Munged_BMI_Meta-analysis_Locke_et_al+UKBiobank_2018_for_LDSC.sumstats.gz
Heritability Results for trait: EUR//Munged_BMI_Meta-analysis_Locke_et_al+UKBiobank_2018_for_LDSC.sumstats.gz
Mean Chi^2 across remaining SNPs: 3.9345
Lambda GC: 2.7889
Intercept: 1.0202 (0.0277)
Ratio: 0.0069 (0.0094)
Total Observed Scale h2: 0.2091 (0.0063)
h2 Z: 33.3
Genetic Correlation Results
LDSC finished running at 2026-05-22 12:43:33.199212
Running LDSC for all files took 0 minutes and 22 seconds


3.946398e-05
BMI
0.2090598
1.020218
686178.2


**Note:** the `ldsc()` function in `GenomicSEM` is designed for bivariate LDSC, so it may print a warning about needing two or more traits. In this practical, that warning can be ignored.


## 7) Bonus: BMI GWAS with LMM

This section uses the already munged BMI summary statistics from Pan-UKBB with a linear mixed model (SAIGE).

BONUS (Time Permitting): LDSC with Sum Stats from Linear Mixed Model GWAS

We will now compare the results of LDSC analyses using BMI GWAS sum stats from the Pan-UKBB project.
These analyses used SAIGE to perform linear mixed models GWAS and include related individuals.

The sum stats have already been munged. The sum stats did not include the sample size, so we have specified the N (419163) when munging.

Run the script lines 165 to 190.

What is the LDSC intercept? Does it suggest significant bias(es) in the BMI GWAS results?

Answer:

 The LDSC intercept is 1.2284 (S.E. = 0.057) - which is significantly >1.

However, LDSC analyses of LMM sum stats may yield intercepts >1
For traits with high SNP heritability
In large samples (such as the UK Biobank, N > 400k)

See Loh, PR., Kichaev, G., Gazal, S. et al. Mixed-model association for biobank-scale datasets. Nat Genet 50, 906–908 (2018). https://doi.org/10.1038/s41588-018-0144-6 for further details.

In this case, we may want to consider some additional points:

Effective N is less than the raw N due to relatedness between some participants.
Could there be residual confounding? Genetic related matrix used as a random effect in LMMs may not control for environmental confounding between family members.

However, note that the attenuation ratio is 0.097 (S.E. = 0.0242), which is totally acceptable.

In [ ]:
munged_lmm_bmi <- file.path(sumstat_dir, "BMI.sumstats.gz")

cat(system(paste0("zcat ", munged_lmm_bmi, " | head"), intern=TRUE), sep='\n')


SNP	N	Z	A1	A2
rs1000050	419163	0.785604482126848	C	T
rs1000073	419163	-0.0144177792690142	A	G
rs1000283	419163	0.254647365736203	A	G
rs1000313	419163	0.840841863887753	G	A
rs1000314	419163	0.845909019382544	G	A
rs1000352	419163	-0.619750547688643	C	T
rs1000370	419163	2.56260288089815	T	G
rs1000417	419163	-3.76968981518794	G	A
rs1000418	419163	-3.77168607864704	G	A


In [ ]:
bmi_lmm_ldsc_out <- file.path(sumstat_dir, "BMI_LMM")

ldsc(
  traits = munged_lmm_bmi,
  sample.prev = NA,
  population.prev = NA,
  ld = ref_data_dir,
  wld = ref_data_dir,
  trait.names = "BMI_lmm",
  ldsc.log = bmi_lmm_ldsc_out
)


Multivariate ld-score regression of 1 traits (EUR//BMI.sumstats.gz) began at: 2026-05-22 12:44:49.143047


Warning message in ldsc(traits = munged_lmm_bmi, sample.prev = NA, population.prev = NA, :
“Our version of ldsc requires 2 or more traits. Please include an additional trait.”


Reading in LD scores
Read in summary statistics [1/1] from: EUR//BMI.sumstats.gz
Out of 97610 SNPs, 97282 remain after merging with LD-score files
Removing 0 SNPs with Chi^2 > 419.163; 97282 remain
Estimating heritability [1/1] for: EUR//BMI.sumstats.gz
Heritability Results for trait: EUR//BMI.sumstats.gz
Mean Chi^2 across remaining SNPs: 3.3545
Lambda GC: 2.4762
Intercept: 1.2284 (0.057)
Ratio: 0.097 (0.0242)
Total Observed Scale h2: 0.2461 (0.0231)
h2 Z: 10.7
Genetic Correlation Results
LDSC finished running at 2026-05-22 12:44:56.196396
Running LDSC for all files took 0 minutes and 7 seconds


0.0005328747
BMI_lmm
0.2460781
1.228402
419163


## End

You have now run the full Colab version of Practical 1 for SNP heritability.
